#  Libraries imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

sns.set_theme(style="whitegrid")

#   Load data

In [ ]:
df = pd.read_csv('data/accident_prediction_india.csv')
df.shape

#   Feature Engineering & Encoding

##  nulls fill 

In [ ]:
df['Traffic Control Presence'] = df['Traffic Control Presence'].fillna('None')
df['Driver License Status'] = df['Driver License Status'].fillna('Unknown')

##  Seperate Target Column

In [ ]:
X = df.drop(columns=['Accident Severity'])
y = df['Accident Severity']

##  Identify Categorical Columns

In [ ]:
categorical_cols = X.select_dtypes(include='object').columns.tolist()
print(categorical_cols)

## Check Cardinality of Categorical Columns

In [ ]:
for col in categorical_cols:
    print(col, '->', X[col].nunique(), 'unique values')

In [ ]:
df['Time of Day'].head(10)

##  Convert "Time of Day" into Time Buckets

In [ ]:
def get_time_bucket(time_str):
    hour = int(time_str.split(':')[0])
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

X['Time Bucket'] = X['Time of Day'].apply(get_time_bucket)
X['Time Bucket'].value_counts()

## Drop Redundant/High-Cardinality Columns

In [ ]:
X = X.drop(columns=['City Name', 'Time of Day'])
X.shape

## One-Hot Encode Categorical Columns

In [ ]:
categorical_cols_final = X.select_dtypes(include='object').columns.tolist()
X_encoded = pd.get_dummies(X, columns=categorical_cols_final, drop_first=True)
X_encoded.shape

## Encode the Target Column
###### Converting the target labels (Minor/Serious/Fatal) into numbers, since scikit-learn models require numeric targets.

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(dict(zip(le.classes_, le.transform(le.classes_))))

## Train/Test Split
###### Splitting data into 80% training and 20% testing, with stratification to preserve severity class balance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
X_train.shape, X_test.shape

## Scale Numeric Features
###### Logistic Regression is sensitive to feature scale, so we standardize all features before training.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Train Baseline Model — Logistic Regression

In [ ]:
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train_scaled, y_train)
y_pred_baseline = baseline_model.predict(X_test_scaled)

print("Baseline Accuracy:", accuracy_score(y_test, y_pred_baseline))
print(classification_report(y_test, y_pred_baseline, target_names=le.classes_))

## Train Tuned Model — Random Forest

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

## Confusion Matrix — Random Forest
###### Visualizing where the model gets confused between severity classes.

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='rocket_r', 
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.tight_layout()
plt.savefig('plots/confusion_matrix_rf.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature Importance — Which factors matter most?

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=X_encoded.columns)
top_features = importances.sort_values(ascending=False).head(10)

plt.figure(figsize=(8,6))
sns.barplot(x=top_features.values, y=top_features.index, hue=top_features.index, 
            palette='rocket', legend=False)
plt.title('Top 10 Most Important Features', fontsize=14, fontweight='bold')
plt.xlabel('Importance')
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig('plots/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Compare Baseline vs Tuned Model

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression (Baseline)', 'Random Forest (Tuned)'],
    'Accuracy': [accuracy_score(y_test, y_pred_baseline), accuracy_score(y_test, y_pred_rf)],
    'F1 Score (weighted)': [f1_score(y_test, y_pred_baseline, average='weighted'), 
                             f1_score(y_test, y_pred_rf, average='weighted')]
})
comparison

## Diagnosing Model Performance — Feature-Target Relationship Check
###### Both models perform close to the random-chance baseline (~33% for 3 classes). Before concluding the models are poorly built, we check whether the dataset's features actually carry a learnable signal for predicting severity.

In [ ]:
for col in ['Alcohol Involvement', 'Lighting Conditions', 'Weather Conditions', 'Speed Limit (km/h)']:
    print(f"\n--- {col} vs Accident Severity ---")
    print(pd.crosstab(df[col], df['Accident Severity'], normalize='index'))

## Conclusion
#### The near-uniform distribution of severity across all feature categories (~33% each) confirms that this dataset's `Accident Severity` labels do not have a statistically learnable relationship with the recorded features. This explains why both models perform at chance level. This is a property of the dataset itself, not a flaw in the modeling approach — model choice, feature engineering, and encoding were all done correctly, but no amount of tuning can extract a signal that isn't present in the data.